[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikbaya/smartbiomed_practicals_2026/blob/main/session3/answers.ipynb)

# Session 3: Complexities of GWAS — Population Structure & Relatedness

**Timing**: ~60 minutes (Parts 1–3 ~35 min; challenges for fast finishers).

**Dataset**: simulated genotypes for 3 discrete populations (Balding–Nichols model) plus admixed
individuals and a few sibling pairs — ~1,000 individuals × 20,000 independent SNPs.
- `y_strat`: a phenotype confounded **purely by ancestry** (environmental — no causal variant).
- `y_clean`: a phenotype with a few **true causal variants** and no confounding.

Everything (PCA, GWAS) is built from scratch with NumPy — no specialised packages.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SETUP — run once. Downloads the data (on Colab) and defines run_gwas + helpers.
# ─────────────────────────────────────────────────────────────────────────────
import os, numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 80, 'font.size': 11})

# Locate the data directory, downloading from GitHub if needed (e.g. on Colab).
import os, urllib.request
_NEED = ['pca_data.npz']
_LFS  = {'gwas_data.npz', 'sumstats_real.npz', 'pca_data.npz', 'finemap_data.npz'}   # Git LFS
def _has_all(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in _NEED)
DATA_DIR = next((d for d in ('../data', 'data') if _has_all(d)), None)
if DATA_DIR is None:
    DATA_DIR = 'data'; os.makedirs(DATA_DIR, exist_ok=True)
    for _f in _NEED:
        _dest = os.path.join(DATA_DIR, _f)
        if os.path.exists(_dest):
            continue
        _base = ('https://media.githubusercontent.com/media' if _f in _LFS
                 else 'https://raw.githubusercontent.com')
        _url = f'{_base}/nikbaya/smartbiomed_practicals_2026/main/data/{_f}'
        print(f'Downloading {_f} from GitHub ...')
        urllib.request.urlretrieve(_url, _dest)

d = np.load(os.path.join(DATA_DIR, 'pca_data.npz'), allow_pickle=True)
G          = d['G'].astype(np.float32)      # (N, M) genotypes 0/1/2 (no missingness here)
pop        = d['pop']                       # 0,1,2 = discrete pops; 3 = admixed
admix_frac = d['admix_frac']                # (N, 3) ancestry proportions
y_strat    = d['y_strat']                   # ancestry-confounded phenotype (no genetics)
y_clean    = d['y_clean']                   # phenotype with true causal variants
causal_idx = d['causal_idx']                # true causal SNP indices for y_clean
rel_pairs  = d['rel_pairs']                 # injected sibling pairs (row indices)
age        = d['age']; sex = d['sex']
pop_names  = d['pop_names']
N, M = G.shape
print(f"Loaded {N:,} individuals × {M:,} SNPs  (pops: {np.bincount(pop)})")

def run_gwas(y, G, covars=None, chunk=5_000):
    """
    Vectorised OLS GWAS: regress phenotype y on each column of G.
    Processes variants in batches to keep peak memory usage low.

    Parameters
    ----------
    y      : (N,)    phenotype (will be mean-centred internally)
    G      : (N, M)  genotype matrix (0/1/2), NaN = missing (treated as mean)
    covars : (N, k)  covariate matrix (optional); age and sex recommended
    chunk  : int     variants per batch (default 5,000)

    Returns
    -------
    betas  : (M,)  per-variant OLS effect size estimate
    ses    : (M,)  standard error of beta
    pvals  : (M,)  two-sided p-value
    """
    N, M = G.shape
    if covars is None:
        C = np.ones((N, 1))
    else:
        C = np.column_stack([np.ones(N), covars])

    # Residualise y on covariates once (cheap)
    Q, _  = np.linalg.qr(C, mode='reduced')
    y_r   = y - Q @ (Q.T @ y)
    ss_y  = float(np.dot(y_r, y_r))
    n_df  = N - C.shape[1] - 1

    betas = np.empty(M); ses = np.empty(M)

    for s in range(0, M, chunk):
        e    = min(s + chunk, M)
        G_c  = G[:, s:e].astype(float)
        mu   = np.nanmean(G_c, axis=0)
        ri, ci = np.where(np.isnan(G_c)); G_c[ri, ci] = mu[ci]   # mean-impute
        G_r  = G_c - Q @ (Q.T @ G_c)                              # residualise
        ss_G = (G_r**2).sum(0)
        b    = G_r.T @ y_r / ss_G
        betas[s:e] = b
        rss  = ss_y - b**2 * ss_G
        ses[s:e]   = np.sqrt(np.clip(rss, 0, None) / n_df / ss_G)

    t_stats = betas / (ses + 1e-300)
    pvals   = 2 * stats.t.sf(np.abs(t_stats), df=n_df)
    return betas, ses, pvals

def qq_plot(pvals, ax=None, color='steelblue', label=''):
    """QQ plot of -log10(p) vs the uniform-null expectation, x-axis truncated."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    n   = len(pvals)
    obs = -np.log10(np.sort(pvals) + 1e-300)
    exp = -np.log10(np.arange(1, n+1) / (n+1))
    ax.scatter(exp, obs, s=4, alpha=0.5, color=color, label=label)
    ax.plot([0, exp.max()], [0, exp.max()], 'r--', lw=1); ax.set_xlim(0, exp.max())
    ax.set_xlabel(r'Expected $-\log_{10}p$'); ax.set_ylabel(r'Observed $-\log_{10}p$')
    if label: ax.legend(fontsize=8)
    return ax

def lambda_gc(pvals):
    chi2 = stats.chi2.isf(np.clip(pvals, 1e-300, 1), df=1)
    return np.median(chi2) / stats.chi2.ppf(0.5, df=1)

print("Setup ready: run_gwas(), qq_plot(), lambda_gc().")


## Part 1: Seeing population stratification

If allele frequencies differ between ancestral groups **and** the phenotype mean differs between
those groups (for *environmental* reasons), then *every* ancestry-informative variant will look
associated — even with no causal effect at all. This is **confounding by population structure**.

`y_strat` was generated with **no genetic effect** — its mean only differs by ancestry. Run a
GWAS and see what the QQ plot / $\lambda_{GC}$ look like.


In [ ]:
# Exercise 1.1: GWAS of the ancestry-confounded phenotype (no covariates)
# YOUR CODE HERE — run_gwas on y_strat with NO covariates
betas, ses, pvals_strat = run_gwas(???, ???)

fig, ax = plt.subplots(figsize=(5, 5))
qq_plot(pvals_strat, ax=ax, label='y_strat, no covariates')
ax.set_title('QQ plot — ancestry-confounded phenotype'); plt.tight_layout(); plt.show()
print(f"lambda_GC = {lambda_gc(pvals_strat):.2f}   (1.0 = no inflation)")
print(f"'Hits' at p<5e-8: {(pvals_strat < 5e-8).sum():,}  — but NOTHING is truly causal!")
print("Q: Why are there so many associations when no variant affects the trait?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

betas, ses, pvals_strat = run_gwas(y_strat, G)
fig, ax = plt.subplots(figsize=(5, 5))
qq_plot(pvals_strat, ax=ax, label='y_strat, no covariates')
ax.set_title('QQ plot — ancestry-confounded phenotype'); plt.tight_layout(); plt.show()
print(f"lambda_GC = {lambda_gc(pvals_strat):.2f}")
print(f"'Hits' at p<5e-8: {(pvals_strat < 5e-8).sum():,}  — all spurious (ancestry confounding).")


## Part 2: Principal Component Analysis (PCA) from scratch

PCA finds the axes of greatest genetic variation across individuals — which, for structured
samples, are the **ancestry axes**. We compute it directly from the standardised genotype matrix
with a singular value decomposition (SVD); no specialised package needed.

If $Z$ is the (N × M) standardised genotype matrix, then $Z = U S V^\top$ and the **principal
component scores** are the columns of $U S$. The top few capture ancestry.


In [ ]:
# Exercise 2.1: PCA from scratch
# Standardise genotypes (mean 0, sd 1 per SNP)
Z = (G - G.mean(0)) / (G.std(0) + 1e-8)

# YOUR CODE HERE
# Economy SVD of Z / sqrt(N); principal-component scores are U * S
U, S, Vt = np.linalg.svd(???, full_matrices=False)
PC = ???                    # (N, k) scores = U * S

# Plot PC1 vs PC2, coloured by population
fig, ax = plt.subplots(figsize=(6, 5))
for k, name in enumerate(pop_names):
    m = pop == k
    ax.scatter(PC[m, 0], PC[m, 1], s=10, alpha=0.6, label=str(name))
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('Genotype PCA'); ax.legend()
plt.tight_layout(); plt.show()
print("Q: Which group is most spread out, and what does that say about its diversity?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

Z = (G - G.mean(0)) / (G.std(0) + 1e-8)
U, S, Vt = np.linalg.svd(Z / np.sqrt(N), full_matrices=False)
PC = U * S
fig, ax = plt.subplots(figsize=(6, 5))
for k, name in enumerate(pop_names):
    m = pop == k
    ax.scatter(PC[m, 0], PC[m, 1], s=10, alpha=0.6, label=str(name))
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('Genotype PCA'); ax.legend()
plt.tight_layout(); plt.show()


## Part 3: Correcting stratification with PCs

Including the top PCs as **covariates** soaks up the ancestry signal, so the spurious associations
disappear and $\lambda_{GC}$ returns to ~1. Crucially, **true** signal survives — re-run the
clean phenotype `y_clean` (which has real causal variants) and check its hits are still there.


In [ ]:
# Exercise 3.1: GWAS with PC covariates
n_pc = 10
covars = PC[:, :n_pc]              # top PCs as covariates

# YOUR CODE HERE — re-run the confounded-phenotype GWAS WITH PC covariates
_, _, pvals_strat_pc = run_gwas(???, ???, ???)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
qq_plot(pvals_strat, ax=axes[0]); axes[0].set_title(f'No PCs (lambda={lambda_gc(pvals_strat):.2f})')
qq_plot(pvals_strat_pc, ax=axes[1], color='#59a14f')
axes[1].set_title(f'With {n_pc} PCs (lambda={lambda_gc(pvals_strat_pc):.2f})')
plt.tight_layout(); plt.show()

# True signal must survive PC correction: GWAS of y_clean with PCs
_, _, pvals_clean = run_gwas(y_clean, G, covars)
print(f"y_clean hits at p<5e-8 (with PCs): {(pvals_clean < 5e-8).sum()}")
print(f"  true causal SNPs recovered: "
      f"{sum(pvals_clean[c] < 5e-8 for c in causal_idx)}/{len(causal_idx)}")
print("Q: PCs removed the false signal but kept the real one — why?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

n_pc = 10
covars = PC[:, :n_pc]
_, _, pvals_strat_pc = run_gwas(y_strat, G, covars)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
qq_plot(pvals_strat, ax=axes[0]); axes[0].set_title(f'No PCs (lambda={lambda_gc(pvals_strat):.2f})')
qq_plot(pvals_strat_pc, ax=axes[1], color='#59a14f')
axes[1].set_title(f'With {n_pc} PCs (lambda={lambda_gc(pvals_strat_pc):.2f})')
plt.tight_layout(); plt.show()
_, _, pvals_clean = run_gwas(y_clean, G, covars)
print(f"y_clean hits at p<5e-8 (with PCs): {(pvals_clean < 5e-8).sum()}")
print(f"  true causal SNPs recovered: {sum(pvals_clean[c] < 5e-8 for c in causal_idx)}/{len(causal_idx)}")


---

## Challenge Questions


### Challenge 1: Admixture in PC space — ~10 min

Admixed individuals carry ancestry from more than one population. Where do they fall in PC space?
Re-plot the PCA, highlighting the admixed group, and colour them by their ancestry proportion.


In [ ]:
# Challenge: admixed individuals in PC space
Z = (G - G.mean(0)) / (G.std(0) + 1e-8)
U, S, Vt = np.linalg.svd(Z / np.sqrt(N), full_matrices=False)
PC = U * S

fig, ax = plt.subplots(figsize=(6.5, 5))
# discrete populations in grey
for k in range(3):
    m = pop == k
    ax.scatter(PC[m, 0], PC[m, 1], s=10, alpha=0.3, color='lightgrey')
    ax.scatter(PC[m, 0].mean(), PC[m, 1].mean(), s=200, marker='X',
               edgecolor='black', label=str(pop_names[k]))
# YOUR CODE HERE: scatter the admixed individuals (pop==3), coloured by proportion of pop 1
adm = pop == 3
sc = ax.scatter(???, ???, c=admix_frac[adm, 0], cmap='viridis', s=25, edgecolor='k', lw=0.2)
plt.colorbar(sc, ax=ax, label='ancestry fraction (Pop1)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('Admixture forms a cline in PC space')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()
print("Q: Why do admixed individuals fall *between* the parental clusters?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

Z = (G - G.mean(0)) / (G.std(0) + 1e-8)
U, S, Vt = np.linalg.svd(Z / np.sqrt(N), full_matrices=False)
PC = U * S
fig, ax = plt.subplots(figsize=(6.5, 5))
for k in range(3):
    m = pop == k
    ax.scatter(PC[m, 0], PC[m, 1], s=10, alpha=0.3, color='lightgrey')
    ax.scatter(PC[m, 0].mean(), PC[m, 1].mean(), s=200, marker='X',
               edgecolor='black', label=str(pop_names[k]))
adm = pop == 3
sc = ax.scatter(PC[adm, 0], PC[adm, 1], c=admix_frac[adm, 0], cmap='viridis',
                s=25, edgecolor='k', lw=0.2)
plt.colorbar(sc, ax=ax, label='ancestry fraction (Pop1)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('Admixture forms a cline in PC space')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()


### Challenge 2: Genomic control vs PCA — ~10 min

**Genomic control (GC)** is a blunter fix: divide every test statistic by $\lambda_{GC}$. It
rescales globally but, unlike PCA, can't tell true polygenic signal from stratification. Apply GC
to the confounded GWAS and compare $\lambda_{GC}$ before, after GC, and after PCA.


In [ ]:
# Challenge: genomic control
chi2 = stats.chi2.isf(np.clip(pvals_strat, 1e-300, 1), df=1)
lam  = lambda_gc(pvals_strat)

# YOUR CODE HERE: divide the chi-squared statistics by lambda, convert back to p-values
chi2_gc = ???
pvals_gc = stats.chi2.sf(chi2_gc, df=1)

print(f"lambda_GC  raw : {lambda_gc(pvals_strat):.2f}")
print(f"lambda_GC  GC  : {lambda_gc(pvals_gc):.2f}   (GC forces median to 1 by construction)")
print(f"hits raw={int((pvals_strat<5e-8).sum())}, after GC={int((pvals_gc<5e-8).sum())}, "
      f"after PCA={int((pvals_strat_pc<5e-8).sum())}")
print("Q: GC and PCA both remove inflation here — when would GC alone be a poor choice?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

chi2 = stats.chi2.isf(np.clip(pvals_strat, 1e-300, 1), df=1)
lam  = lambda_gc(pvals_strat)
chi2_gc = chi2 / lam
pvals_gc = stats.chi2.sf(chi2_gc, df=1)
print(f"lambda_GC  raw : {lambda_gc(pvals_strat):.2f}")
print(f"lambda_GC  GC  : {lambda_gc(pvals_gc):.2f}")
print(f"hits raw={int((pvals_strat<5e-8).sum())}, after GC={int((pvals_gc<5e-8).sum())}, "
      f"after PCA={int((pvals_strat_pc<5e-8).sum())}")
# GC rescales everything globally; it cannot separate true polygenic signal from stratification,
# and over-corrects real associations. PCA models the structure directly.


### Challenge 3: Relatedness and the GRM — ~12 min

Close relatives share long genomic segments, which (like ancestry) violates the independence GWAS
assumes. The **genetic relatedness matrix** $\text{GRM} = \frac{1}{M} Z Z^\top$ quantifies it.
A few **sibling pairs** were hidden in the data — find them, and think about why mixed models exist.


In [ ]:
# Challenge: genetic relatedness matrix
Z = (G - G.mean(0)) / (G.std(0) + 1e-8)

# YOUR CODE HERE: GRM = Z Z^T / M  (diagonal ≈ 1 = self; off-diagonal ≈ kinship*2)
GRM = ???

off = GRM[np.triu_indices(N, k=1)]
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(off, bins=120, color='steelblue'); ax.set_yscale('log')
ax.axvline(0.5, color='red', ls='--', label='sib/parent ≈ 0.5')
ax.set_xlabel('GRM off-diagonal (relatedness)'); ax.set_ylabel('pairs (log)'); ax.legend()
ax.set_title('Most pairs ≈ 0; a few related pairs stand out'); plt.tight_layout(); plt.show()

print("Injected sibling pairs (should be ≈ 0.5):")
for i, j in rel_pairs:
    print(f"  rows {i},{j}: GRM = {GRM[i, j]:.2f}")
print("Q: Why does ignoring relatedness inflate test statistics, and how do mixed models help?")


In [ ]:
# @title Solution { display-mode: "form" }
# ▼ Click ▶ to reveal solution

Z = (G - G.mean(0)) / (G.std(0) + 1e-8)
GRM = Z @ Z.T / M
off = GRM[np.triu_indices(N, k=1)]
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(off, bins=120, color='steelblue'); ax.set_yscale('log')
ax.axvline(0.5, color='red', ls='--', label='sib/parent ≈ 0.5')
ax.set_xlabel('GRM off-diagonal (relatedness)'); ax.set_ylabel('pairs (log)'); ax.legend()
ax.set_title('Most pairs ≈ 0; a few related pairs stand out'); plt.tight_layout(); plt.show()
print("Injected sibling pairs (should be ≈ 0.5):")
for i, j in rel_pairs:
    print(f"  rows {i},{j}: GRM = {GRM[i, j]:.2f}")
# Relatives share segments → correlated residuals → inflated statistics. A linear mixed model
# puts the GRM in the covariance of a random effect, accounting for this structure.
